In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

# =========================
# LOAD DATA
# =========================
df = pd.read_csv('../../data/narc_data/merged_dataset_final_all.csv')

# =========================
# FEATURES & TARGETS
# =========================
X = df[['ph','organic_matter','total_nitrogen','potassium','p2o5',
    'zinc','sand','clay','slit','boron']]

y = df[['UREA1', 'DAP', 'MOP']]

# =========================
# TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# =========================
# PIPELINE (Scaling + MLR)
# =========================
model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', MultiOutputRegressor(LinearRegression()))
])

# =========================
# TRAIN
# =========================
model.fit(X_train, y_train)

# =========================
# PREDICT
# =========================
y_train_pred = model.predict(X_train)
y_test_pred  = model.predict(X_test)

# =========================
# EVALUATION FUNCTION
# =========================
def evaluate(y_true, y_pred, label):
    print(f"\n===== {label} Set Evaluation =====")
    results = []

    for i, col in enumerate(y.columns):
        rmse = np.sqrt(
            mean_squared_error(
                y_true.iloc[:, i],
                y_pred[:, i]
            )
        )
        mae = mean_absolute_error(
            y_true.iloc[:, i],
            y_pred[:, i]
        )
        r2 = r2_score(
            y_true.iloc[:, i],
            y_pred[:, i]
        )

        results.append([rmse, mae, r2])

    df_res = pd.DataFrame(
        results,
        columns=['RMSE', 'MAE', 'R2'],
        index=y.columns
    )
    print(df_res)

    print("\n-- Overall (macro average) --")
    print("RMSE:", np.mean(df_res['RMSE']))
    print("MAE :", np.mean(df_res['MAE']))
    print("R2  :", np.mean(df_res['R2']))

# =========================
# RESULTS
# =========================
evaluate(y_train, y_train_pred, "Training")
evaluate(y_test, y_test_pred, "Testing")



===== Training Set Evaluation =====
           RMSE       MAE        R2
UREA1  0.237221  0.180190  0.567946
DAP    0.387186  0.321471  0.708347
MOP    0.336893  0.211956  0.358184

-- Overall (macro average) --
RMSE: 0.32043339609885746
MAE : 0.23787232558978913
R2  : 0.5448256714018802

===== Testing Set Evaluation =====
           RMSE       MAE        R2
UREA1  0.236859  0.180010  0.572800
DAP    0.387878  0.321144  0.710573
MOP    0.338138  0.211021  0.352972

-- Overall (macro average) --
RMSE: 0.3209583370320898
MAE : 0.23739152117927811
R2  : 0.5454483900366018


In [10]:
import joblib
import os

os.makedirs("saved_models", exist_ok=True)

joblib.dump(model, "saved_models/mlr_fertilizer_pipeline.pkl")

print("✅ Model and scaler saved successfully")


✅ Model and scaler saved successfully


In [11]:
import joblib

model = joblib.load("saved_models/mlr_fertilizer_pipeline.pkl")

print("✅ Model loaded successfully")


✅ Model loaded successfully


In [20]:
new_sample = pd.DataFrame([{
    'ph': 6.4,
    'organic_matter': 10,
    'total_nitrogen': 100,
    'potassium': 115,
    'p2o5': 42,
    'zinc': 1.3,
    'sand': 40,
    'clay': 30,
    'slit': 30,
    'boron': 0.6
}])

prediction = model.predict(new_sample)

pred_df = pd.DataFrame(
    prediction,
    columns=['UREA1', 'DAP', 'MOP']
)

print("\n===== Fertilizer Recommendation =====")
print(pred_df)



===== Fertilizer Recommendation =====
        UREA1          DAP         MOP
0 -536.483516  1187.029575  138.107609
